# Part 3C: object-detection training smoke test

Run with the separate `.venv-ml` environment from the repository root. This notebook exercises the reusable training package on one selected training sample and two validation samples in `best_effort` reproducibility mode; it is not a performance benchmark and never constructs or evaluates the test partition.

In [1]:
from pathlib import Path
import json
import sys
import torch

project_root = Path.cwd()
if not (project_root / "src").is_dir():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

from scenarioforge.training.experiment import DetectionExperiment

config_path = project_root / "configs" / "object_detection_smoke.json"
experiment = DetectionExperiment.from_config_file(config_path)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Experiment directory:", experiment.run_dir)

PyTorch: 2.15.0+cu132
CUDA available: True
Experiment directory: experiments\20261103T163030Z-e652390e


## Inspect a real DataLoader batch

Images and targets should remain on CPU in the DataLoader; the trainer transfers one batch at a time.

In [2]:
batch_report = experiment.inspect_batch()
print(json.dumps(batch_report, indent=2))
assert batch_report["batch_size"] == 1
assert batch_report["images"][0]["image_dtype"] == "torch.float32"
assert batch_report["images"][0]["boxes_shape"][1] == 4

{
  "batch_size": 1,
  "device": "cuda",
  "images": [
    {
      "image_shape": [
        3,
        900,
        1600
      ],
      "image_dtype": "torch.float32",
      "image_device": "cpu",
      "target_keys": [
        "area",
        "boxes",
        "image_id",
        "iscrowd",
        "labels"
      ],
      "boxes_shape": [
        17,
        4
      ],
      "boxes_dtype": "torch.float32",
      "labels_shape": [
        17
      ],
      "labels_dtype": "torch.int64",
      "box_count": 17
    }
  ],
  "cuda_allocated_bytes": 0,
  "cuda_reserved_bytes": 0
}


## Instantiate Faster R-CNN

The configured detector uses torchvision COCO_V1 initialization and a ScenarioForge head with ten foreground classes plus background.

In [3]:
model = experiment.create_model()
parameters = list(model.parameters())
print("Architecture:", model.architecture)
print("Pretrained weights:", model.pretrained_weights)
print("Output classes (including background):", model.detector.roi_heads.box_predictor.cls_score.out_features)
print("Parameters:", sum(parameter.numel() for parameter in parameters))
print("Trainable parameters:", sum(parameter.numel() for parameter in parameters if parameter.requires_grad))
assert model.detector.roi_heads.box_predictor.cls_score.out_features == 11

Architecture: fasterrcnn_mobilenet_v3_large_320_fpn
Pretrained weights: COCO_V1
Output classes (including background): 11
Parameters: 18976354
Trainable parameters: 18976354


## Forward/backward and checkpoint smoke test

This performs one optimizer step on a disposable model, reports individual loss components and peak CUDA allocation, then reloads the saved checkpoint. It restores RNG and loader-generator state so the following epoch smoke run starts independently.

In [4]:
smoke_result = experiment.run_one_batch_smoke_test()
print(json.dumps(smoke_result, indent=2, sort_keys=True))
assert smoke_result["checkpoint_reload_verified"]

D:\1Projectss\2027Application-projects\ScenarioForge\.venv-ml\Lib\site-packages\torchvision\_autograd_registrations.py:27: UserWarning: roi_align_backward_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation.
  torch._prims_common.alert_not_deterministic("roi_align_backward_kernel")


{
  "checkpoint": "experiments\\20261103T163030Z-e652390e\\checkpoint_smoke.pth",
  "checkpoint_reload_verified": true,
  "cuda_peak_allocated_bytes": 386103808,
  "device": "cuda",
  "losses": {
    "loss_box_reg": 0.35033512115478516,
    "loss_classifier": 2.7271170616149902,
    "loss_objectness": 0.13077035546302795,
    "loss_rpn_box_reg": 0.10800273716449738,
    "total_loss": 3.316225290298462
  }
}


## Short training and validation

The smoke configuration trains for one epoch on one selected train sample and validates on two validation samples. The test partition is not passed to the trainer.

In [5]:
history = experiment.train()
print("Epochs:", len(history))
print("Best validation mAP@0.50:0.95:", experiment.trainer.best_validation_ap)
print("Best checkpoint:", experiment.run_dir / "checkpoint_best.pth")
assert (experiment.run_dir / "checkpoint_best.pth").is_file()
assert (experiment.run_dir / "checkpoint_last.pth").is_file()

Epochs: 1
Best validation mAP@0.50:0.95: 0.0
Best checkpoint: experiments\20261103T163030Z-e652390e\checkpoint_best.pth


## Checkpoint evaluation

Reload the frozen best checkpoint and evaluate on validation data. No test metrics are computed in this notebook.

In [6]:
validation_metrics = experiment.evaluate_checkpoint(
    experiment.run_dir / "checkpoint_best.pth",
    split_name="validation",
)
print(json.dumps(validation_metrics, indent=2, sort_keys=True))
assert validation_metrics["split"] == "validation"
assert "map_50" in validation_metrics and "map_75" in validation_metrics

{
  "image_count": 2,
  "iou_thresholds": [
    0.5,
    0.55,
    0.6,
    0.65,
    0.7,
    0.75,
    0.8,
    0.85,
    0.9,
    0.95
  ],
  "map_50": 0.0,
  "map_50_95": 0.0,
  "map_75": 0.0,
  "metric_implementation": "torchmetrics.detection.MeanAveragePrecision/faster_coco_eval",
  "per_class": {
    "barrier": {
      "average_precision": null,
      "ground_truth_objects": 0,
      "precision_iou50": null,
      "predictions": 25,
      "predictions_at_score_threshold": 0,
      "recall_iou50": null
    },
    "bicycle": {
      "average_precision": null,
      "ground_truth_objects": 0,
      "precision_iou50": null,
      "predictions": 20,
      "predictions_at_score_threshold": 0,
      "recall_iou50": null
    },
    "bus": {
      "average_precision": null,
      "ground_truth_objects": 0,
      "precision_iou50": null,
      "predictions": 3,
      "predictions_at_score_threshold": 0,
      "recall_iou50": null
    },
    "car": {
      "average_precision": 0.0,
      "